# Day 1 — Logic, SAT & SMT with Z3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/01_day1_logic_sat_smt.ipynb)

Runs the **actual** Day-1 worked examples from [`day01/examples/`](https://github.com/ttj/fmaiv/tree/main/day01/examples) — the same files CI checks — so the notebook and the repo never drift. Works in **Google Colab** (the setup cell clones the repo and installs Z3) and in **GitHub Codespaces / the course image** (everything is already there, so setup is a no-op).

## Setup

In [ ]:
# --- Setup: find the course repo (clone it on Colab), define run helpers ------
# Idempotent: in GitHub Codespaces / the course image the repo and tools are
# already present, so the installs in the next cell are skipped. On Google Colab
# this clones the repo once. Re-running is safe.
import os, sys, re, subprocess, shutil, pathlib

def sh(cmd):
    """Run a shell command, streaming output; raise on failure."""
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

def run(cmd, expect=None):
    """Run a command, show its output, and (optionally) assert a verdict regex
    appears -- so this notebook self-checks exactly like CI (check_examples.sh)."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or '') + (r.stderr or '')
    print(out.rstrip())
    if expect is not None:
        assert re.search(expect, out), f'FAILED: expected /{expect}/ in output'
        print(f'  [ok] matched /{expect}/')
    elif r.returncode != 0:
        raise RuntimeError(f'command exited {r.returncode}')
    return out

def find_repo_root(marker='day01/examples'):
    for d in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
        if (d / marker).is_dir():
            return d
    return None

REPO = find_repo_root()
if REPO is None:                       # Colab: no repo on disk -> clone it once
    if not pathlib.Path('fmaiv').exists():
        sh('git clone --depth 1 https://github.com/ttj/fmaiv')
    REPO = pathlib.Path('fmaiv').resolve()
os.chdir(REPO)
assert (REPO / 'day01' / 'examples').is_dir(), 'unexpected repo layout'
print('Course repo:', REPO)

In [ ]:
# Z3 -- Python bindings (+ a `z3` CLI in Codespaces/the image). On Colab this
# pip-installs the bindings in seconds; the .smt2 cell below falls back to the
# Python API when the `z3` CLI is not on PATH (the pip wheel ships no CLI).
try:
    import z3
except ImportError:
    sh(f'"{sys.executable}" -m pip install -q z3-solver'); import z3
HAVE_Z3_CLI = shutil.which('z3') is not None
print('Z3', z3.get_version_string(), '| z3 CLI on PATH:', HAVE_Z3_CLI)

## 1. Core Z3 examples
SAT/SMT smoke test, an UNSAT pigeonhole proof, a bounded counter invariant, validity-via-`assert-not`, and program synthesis as exists-forall solving.

In [ ]:
# The real files under day01/examples/ -- each prints its result and exits 0 on success.
for f in ['z3_smoke.py', 'z3_pigeonhole.py', 'z3_counter_bounded.py',
          'z3_entailment.py', 'z3_synthesis.py']:
    print(f'\n===== day01/examples/{f} =====')
    run(f'"{sys.executable}" day01/examples/{f}')

## 2. SMT-LIB (`.smt2`) files
The textual SMT-LIB 2 input format. Uses the `z3` CLI when present (Codespaces), else the Python API.

In [ ]:
for f in ['z3_smt_basics.smt2', 'z3_smtlib_demo.smt2']:
    p = f'day01/examples/{f}'
    print(f'\n===== {p} =====')
    if HAVE_Z3_CLI:
        run(f'z3 {p}', expect='sat')
    else:
        s = z3.Solver(); s.from_file(p); print('check:', s.check())

## 3. Constraint puzzles
Classic puzzles encoded as constraints (each ships a `*_starter.py` to fill in yourself).

In [ ]:
for f in ['sudoku.py', 'nqueens.py', 'magic_square.py', 'kenken.py']:
    print(f'\n===== day01/examples/puzzles/{f} =====')
    run(f'"{sys.executable}" day01/examples/puzzles/{f}')

### Next steps
- **Slides:** [Day 1 — Foundations](https://ttj.github.io/fmaiv/day01.html)
- **Try it yourself:** each example has a `*_starter.py` (e.g. `day01/examples/z3_entailment_starter.py`) — open it, fill in the blanks, and re-run the cell above.
- **No install, in the browser:** the [Z3 guide](https://microsoft.github.io/z3guide/).
- **Next:** `02_day2_model_checking.ipynb` (NuSMV); for state/BDD visualization use the [smvis web app](https://bit.ly/fmaiv_smvis).